In [1]:
from IPython.display import display

import hashlib
import json
import os
import uuid as GenUUID


import numpy as np
import pandas as pd

import duckdb
from duckdb.typing import *

data_dir = '/home/ekansa/oc-data'

tables = [
    ('samps', os.path.join(data_dir, 'isamples_oc_test.parquet'),),
    ('agents', os.path.join(data_dir, 'isamples_oc_test_persons.parquet'),),
    ('asserts', os.path.join(data_dir, 'isamples_oc_test_asserts.parquet'),),
    ('oc_man', os.path.join(data_dir, 'oc_all_manifest.parquet'),),
    ('test', os.path.join(data_dir, 'test_10.parquet'),),
]
for tab, path in tables:
    sql = f"""
    CREATE TABLE {tab} AS
    SELECT * FROM '{path}';
    """
    duckdb.sql(sql)
     


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [2]:
PID_SAMPSITE = "concat('https://', isam_sampling_site_uri)"
PID_GEOLOC_SAMP = "get_deterministic_id(item__geo_source_uri, 'geoloc_')"
PID_GEOLOC_SITE = f"get_deterministic_id({PID_SAMPSITE}, 'geoloc_')"
PID_SAMP = "use_for_pid(persistent_ark, uri)"
PID_SAMPEVENT = "get_deterministic_id(uri, 'sampevent_')"


In [3]:
# De-duplicate ARKS
sql = """
SELECT count(uuid) AS dup_ark_count, persistent_ark 
FROM samps 
WHERE persistent_ark IS NOT NULL 
GROUP BY persistent_ark 
ORDER BY dup_ark_count DESC, persistent_ark 
"""
display(duckdb.sql(sql).show(max_rows=10))

┌───────────────┬──────────────────────┐
│ dup_ark_count │    persistent_ark    │
│     int64     │       varchar        │
├───────────────┼──────────────────────┤
│             1 │ ark:/28722/k2000027w │
│             1 │ ark:/28722/k2000028c │
│             1 │ ark:/28722/k2000029v │
│             1 │ ark:/28722/k2000030z │
│             1 │ ark:/28722/k2000031f │
│             · │          ·           │
│             · │          ·           │
│             · │          ·           │
│             1 │ ark:/28722/k20869227 │
│             1 │ ark:/28722/k2086923q │
│             1 │ ark:/28722/k20869246 │
│             1 │ ark:/28722/k2086925p │
│             1 │ ark:/28722/k20869265 │
├───────────────┴──────────────────────┤
│    ? rows (>9999 rows, 10 shown)     │
└──────────────────────────────────────┘



None

In [4]:
duckdb.sql('SELECT COUNT(uuid) FROM samps')


┌─────────────┐
│ count(uuid) │
│    int64    │
├─────────────┤
│     1050555 │
└─────────────┘

In [5]:
duckdb.sql('SELECT COUNT(*) FROM test')

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│          274 │
└──────────────┘

In [6]:
display(duckdb.sql('DESCRIBE SELECT * FROM test').show(max_rows=100))

┌─────────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│       column_name       │ column_type │  null   │   key   │ default │  extra  │
│         varchar         │   varchar   │ varchar │ varchar │ varchar │ varchar │
├─────────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ pid                     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ tcreated                │ INTEGER     │ YES     │ NULL    │ NULL    │ NULL    │
│ tmodified               │ INTEGER     │ YES     │ NULL    │ NULL    │ NULL    │
│ otype                   │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ s                       │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ p                       │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ o                       │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ n                       │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ altids        

None

In [7]:
display(duckdb.sql('DESCRIBE SELECT * FROM samps').show(max_rows=100))

┌──────────────────────────┬──────────────────────────┬─────────┬─────────┬─────────┬─────────┐
│       column_name        │       column_type        │  null   │   key   │ default │  extra  │
│         varchar          │         varchar          │ varchar │ varchar │ varchar │ varchar │
├──────────────────────────┼──────────────────────────┼─────────┼─────────┼─────────┼─────────┤
│ uuid                     │ UUID                     │ YES     │ NULL    │ NULL    │ NULL    │
│ slug                     │ VARCHAR                  │ YES     │ NULL    │ NULL    │ NULL    │
│ label                    │ VARCHAR                  │ YES     │ NULL    │ NULL    │ NULL    │
│ sort                     │ VARCHAR                  │ YES     │ NULL    │ NULL    │ NULL    │
│ published                │ TIMESTAMP WITH TIME ZONE │ YES     │ NULL    │ NULL    │ NULL    │
│ revised                  │ TIMESTAMP WITH TIME ZONE │ YES     │ NULL    │ NULL    │ NULL    │
│ updated                  │ TIMESTAMP W

None

In [8]:
display(duckdb.sql('DESCRIBE SELECT * FROM asserts').show(max_rows=100))

┌──────────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│       column_name        │ column_type │  null   │   key   │ default │  extra  │
│         varchar          │   varchar   │ varchar │ varchar │ varchar │ varchar │
├──────────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ uuid                     │ UUID        │ YES     │ NULL    │ NULL    │ NULL    │
│ label                    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ path                     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ uri                      │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ predicate_equiv_ld_uri   │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ predicate_equiv_ld_label │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ object_equiv_ld_uri      │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ object_equiv_ld_label    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ pr

None

In [9]:
display(duckdb.sql('DESCRIBE SELECT * FROM agents').show(max_rows=100))


┌─────────────────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│           column_name           │ column_type │  null   │   key   │ default │  extra  │
│             varchar             │   varchar   │ varchar │ varchar │ varchar │ varchar │
├─────────────────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ uuid                            │ UUID        │ YES     │ NULL    │ NULL    │ NULL    │
│ label                           │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ path                            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ uri                             │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ project_uuid                    │ UUID        │ YES     │ NULL    │ NULL    │ NULL    │
│ project_label                   │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ project_uri                     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ predicat

None

In [10]:
# Define some functions to use in Duckdb
def get_deterministic_id(values: str, prefix='') -> str:
    values = str(values)
    hash_obj = hashlib.sha1()
    hash_obj.update(values.encode('utf-8'))
    hash_id =  hash_obj.hexdigest()
    return f'{prefix}{hash_id}'

def is_obscured(geo_spec:int) -> bool:
    if geo_spec is None:
        return False
    if geo_spec < 0:
        return True
    return False

def use_for_pid(best_id: str, fallback_id:str) -> str:
    best_id = str(best_id)
    if not best_id or best_id == 'None':
        return fallback_id
    return best_id

def make_alt_id_list(id_1:str='', id_2:str='', id_3:str='', id_4:str='') -> list:
    id_list = []
    if id_1:
        id_list.append(id_1)
    if id_2:
        id_list.append(id_2)
    if id_3:
        id_list.append(id_3)
    if id_4:
        id_list.append(id_4)
    return id_list


In [11]:
duckdb.create_function(
    "get_deterministic_id", 
    get_deterministic_id, 
    [VARCHAR, VARCHAR], VARCHAR, 
    null_handling="special"
)

In [12]:
duckdb.create_function(
    "is_obscured", 
    is_obscured, 
    [BIGINT], BOOLEAN, 
    null_handling="special"
)

In [13]:
duckdb.create_function(
    "use_for_pid", 
    use_for_pid, 
    [VARCHAR, VARCHAR], VARCHAR, 
    null_handling="special"
)

In [14]:
duckdb.create_function(
    "make_alt_id_list", 
    make_alt_id_list, 
    [VARCHAR, VARCHAR, VARCHAR, VARCHAR], 'VARCHAR[]', 
    null_handling="special"
)

In [15]:
# Copy the test table to make an output table
sql = 'DROP TABLE IF EXISTS out'
duckdb.sql(sql)
sql = f"""
    CREATE TABLE out AS
    SELECT * FROM test LIMIT 0;
"""
duckdb.sql(sql)
sql = "CREATE UNIQUE INDEX pid_idx ON out (pid);"
duckdb.sql(sql)
display(duckdb.sql('DESCRIBE SELECT * FROM out').show(max_rows=100))

┌─────────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│       column_name       │ column_type │  null   │   key   │ default │  extra  │
│         varchar         │   varchar   │ varchar │ varchar │ varchar │ varchar │
├─────────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ pid                     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ tcreated                │ INTEGER     │ YES     │ NULL    │ NULL    │ NULL    │
│ tmodified               │ INTEGER     │ YES     │ NULL    │ NULL    │ NULL    │
│ otype                   │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ s                       │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ p                       │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ o                       │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ n                       │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ altids        

None

In [16]:
NEW_KEYS = [
    (PID_SAMPSITE, 'PID_SAMPSITE',),
    (PID_GEOLOC_SAMP, 'PID_GEOLOC_SAMP',),
    (PID_GEOLOC_SITE, 'PID_GEOLOC_SITE',),
    (PID_SAMP, 'PID_SAMP',),
    (PID_SAMPEVENT, 'PID_SAMPEVENT',),
]
for t_col, col in NEW_KEYS:
    sql = f"ALTER TABLE samps ADD COLUMN {col} VARCHAR DEFAULT NULL"
    duckdb.sql(sql)
    sql = f"UPDATE samps SET {col} = {t_col}"
    duckdb.sql(sql)

In [17]:
# Great Geospatial location entities. These are used directly for the material samples
sql = f"""
INSERT OR IGNORE INTO out (
    pid,
    latitude,
    longitude,
    obfuscated,
    elevation,
    otype
) SELECT
    PID_GEOLOC_SAMP, 
    ANY_VALUE(item__latitude)::double,
    ANY_VALUE(item__longitude)::double,
    is_obscured(ANY_VALUE(item__geo_specificity)),
    NULL,
    'GeospatialCoordLocation'
    FROM samps
    WHERE item__geo_source_uri  IS NOT NULL
    AND PID_GEOLOC_SAMP NOT IN
    (SELECT pid FROM out)
    GROUP BY PID_GEOLOC_SAMP
"""
duckdb.sql(sql)
sql = """
SELECT
    pid,
    latitude,
    longitude,
    obfuscated,
    elevation,
    otype
FROM out
WHERE otype = 'GeospatialCoordLocation'
"""
display(duckdb.sql(sql).show(max_rows=100))

┌─────────────────────────────────────────────────┬────────────────────┬─────────────────────┬────────────┬───────────┬─────────────────────────┐
│                       pid                       │      latitude      │      longitude      │ obfuscated │ elevation │          otype          │
│                     varchar                     │       double       │       double        │  boolean   │  varchar  │         varchar         │
├─────────────────────────────────────────────────┼────────────────────┼─────────────────────┼────────────┼───────────┼─────────────────────────┤
│ geoloc_7bb72647e680774eb660f0a3ee62dcbce2cfe109 │           30.64804 │           -97.60076 │ true       │ NULL      │ GeospatialCoordLocation │
│ geoloc_95d00ebb1c6ac6fa48cea4fc25a8015387dc1e33 │  47.94861666055502 │  -77.77942752492783 │ false      │ NULL      │ GeospatialCoordLocation │
│ geoloc_7a845cdf64e5ce3d84720831cc4e0feed9e2453d │            55.7874 │             -6.4336 │ false      │ NULL      │ Geos

None

In [18]:
# Great Geospatial location entities for the "Sampling Sites". We can get better data from Open Context, but we'll get
# pretty good sampling site location data this way.

sql = f"""
INSERT OR IGNORE INTO out (
    pid,
    latitude,
    longitude,
    obfuscated,
    elevation,
    otype
) SELECT 
    PID_GEOLOC_SITE, 
    favg(item__latitude)::double,
    favg(item__longitude)::double,
    is_obscured(min(item__geo_specificity)),
    NULL,
    'GeospatialCoordLocation'
    FROM samps
    WHERE isam_sampling_site_uri  IS NOT NULL
    AND PID_GEOLOC_SITE NOT IN
    (SELECT pid FROM out)
    GROUP BY PID_GEOLOC_SITE
"""
duckdb.sql(sql)

In [19]:
# Add the Sampling Sites
sql = f"""
INSERT OR IGNORE INTO out (
    pid,
    description,
    label,
    place_name,
    is_part_of,
    otype
) SELECT 
    PID_SAMPSITE, 
    concat_ws(' ', 'A sampling site documented by: ', ANY_VALUE(project_label)),
    ANY_VALUE(isam_sampling_site_label),
    [ANY_VALUE(path_to___1), ANY_VALUE(path_to___2)],
    NULL,
    'SamplingSite'
    FROM samps
    WHERE isam_sampling_site_uri  IS NOT NULL
    AND PID_SAMPSITE NOT IN
    (SELECT pid FROM out)
    GROUP BY PID_SAMPSITE
"""
duckdb.sql(sql)

In [20]:
# Make the edges relating sampling sites to sample site location

def make_s_p_o_edge_row(s_col, p_val, o_col, p_col=None, source_tab='samps', group_by_cols=None):

    sql = "DROP TABLE IF EXISTS spo"
    duckdb.sql(sql)

    if p_col:        
        if not group_by_cols:
            group_by_cols = f"{s_col}, {p_col}, {o_col}"

        sql = f"""
        CREATE TABLE spo AS
        SELECT 
            {s_col} AS s,
            {p_col} AS p,
            {o_col} AS o,
            '_edge_' AS otype
            FROM {source_tab}
            WHERE {s_col} IS NOT NULL
            AND {p_col} IS NOT NULL
            AND {o_col} IS NOT NULL
            GROUP BY {group_by_cols}
        """
    else:

        if not group_by_cols:
            group_by_cols = f"{s_col}, {o_col}"
        
        sql = f"""
        CREATE TABLE spo AS
        SELECT 
            {s_col} AS s,
            '{p_val}' AS p,
            {o_col} AS o,
            '_edge_' AS otype
            FROM {source_tab}
            WHERE {s_col} IS NOT NULL
            AND {o_col} IS NOT NULL
            GROUP BY {group_by_cols}
        """
    # Now do the SQL to make the temprary table
    duckdb.sql(sql)

    # Use the temporary table to make the insert to the output
    sql = """
    INSERT OR IGNORE INTO out  (
        pid,
        s,
        p,
        o,
        otype
    ) SELECT
        get_deterministic_id(concat(s, p, o), 'edge_') AS pid,
        s,
        p,
        o,
        otype
        FROM spo
        WHERE s IS NOT NULL
        AND s IN (SELECT pid FROM out)
        AND p IS NOT NULL
        AND o IS NOT NULL
        AND o IN (SELECT pid FROM out)
        AND get_deterministic_id(concat(s, p, o), 'edge_') NOT IN
        (SELECT pid FROM out)
        GROUP BY s , p, o, otype
    """
    duckdb.sql(sql)


In [21]:
# Add Material Sample Records
sql = f"""
INSERT OR IGNORE INTO out  (
    pid,
    altids,
    alternate_identifiers,
    complies_with,
    dc_rights,
    description,
    label,
    sample_identifier,
    otype
) SELECT 
    PID_SAMP,
    [ANY_VALUE(uri), ANY_VALUE(persistent_ark), ANY_VALUE(persistent_doi)] AS altids,
    [ANY_VALUE(uri), ANY_VALUE(persistent_ark), ANY_VALUE(persistent_doi)] AS altids,
    NULL,
    NULL,
    concat_ws(' ', 'Open Context published sample record from:', path, 'of:', ANY_VALUE(item_class_label)),
    ANY_VALUE(label),
    ANY_VALUE(label) AS sample_identifier,
    'MaterialSampleRecord'
    FROM samps
    WHERE PID_SAMP NOT IN (SELECT pid FROM out)
    AND PID_SAMP IS NOT NULL
    GROUP BY PID_SAMP, PATH

"""
duckdb.sql(sql)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [22]:
sql = """
SELECT
    pid,
    description,
    label,
    otype
FROM out
WHERE otype = 'MaterialSampleRecord'
"""
display(duckdb.sql(sql).show(max_rows=100))

┌─────────────────────────────────┬───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬───────────────┬──────────────────────┐
│               pid               │                                                                                description                                                                                │     label     │        otype         │
│             varchar             │                                                                                  varchar                                                                                  │    varchar    │       varchar        │
├─────────────────────────────────┼───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┼───────────────┼──────────────────────┤
│ ark:/28722

None

In [23]:
# Make sure the Open Context manifest table has https prefixes
sql = "UPDATE oc_man SET uri = concat('https://', uri) WHERE uri not like 'https://%'"
duckdb.sql(sql)


In [24]:
sql = """
INSERT OR IGNORE INTO out  (
    pid,
    label,
    scheme_name,
    scheme_uri,
    otype
) SELECT DISTINCT
    object_equiv_ld_uri, 
    object_equiv_ld_label, 
    con_man.label, 
    con_man.uri,
    'IdentifiedConcept'
    FROM asserts
    INNER JOIN oc_man ON object_equiv_ld_uri = oc_man.uri
    INNER JOIN oc_man AS con_man ON oc_man.context_uuid = con_man.uuid
    WHERE object_equiv_ld_uri IS NOT NULL
    AND object_equiv_ld_uri NOT IN (SELECT pid FROM out)
"""

duckdb.sql(sql)

In [25]:
sql = """
INSERT OR IGNORE INTO out  (
    pid,
    label,
    scheme_name,
    scheme_uri,
    otype
) SELECT DISTINCT
    object_uri, 
    object_label, 
    con_man.label, 
    con_man.uri,
    'IdentifiedConcept'
    FROM asserts
    INNER JOIN oc_man ON object_uri = oc_man.uri
    INNER JOIN oc_man AS con_man ON oc_man.project_uuid = con_man.uuid
    WHERE object_equiv_ld_uri IS NOT NULL
    AND object_equiv_ld_uri NOT IN (SELECT pid FROM out)
"""
duckdb.sql(sql)

In [26]:
sql = """
SELECT
    pid,
    label,
    scheme_name,
    scheme_uri,
    otype
FROM out
WHERE otype = 'IdentifiedConcept'
"""
display(duckdb.sql(sql).show(max_rows=100))

┌─────────────────────────────────────────────────────────────────┬───────────────────────────────────┬───────────────────────────────────────────────────────────┬──────────────────────────────────────┬───────────────────┐
│                               pid                               │               label               │                        scheme_name                        │              scheme_uri              │       otype       │
│                             varchar                             │              varchar              │                          varchar                          │               varchar                │      varchar      │
├─────────────────────────────────────────────────────────────────┼───────────────────────────────────┼───────────────────────────────────────────────────────────┼──────────────────────────────────────┼───────────────────┤
│ https://n2t.net/ark:/99152/p0m64td7h8h                          │ Middle Kingdom, Egypt             │ EAME

None

In [27]:
# Add Sampling Events Rows
sql = f"""
INSERT OR IGNORE INTO out  (
    pid,
    label,
    project,
    otype
) SELECT 
    PID_SAMPEVENT,
    concat_ws(' ', 'Sampling event for:', ANY_VALUE(label)),
    ANY_VALUE(project_label),
    'SamplingEvent'
    FROM samps
    WHERE PID_SAMPEVENT NOT IN (SELECT pid FROM out)
    GROUP BY PID_SAMPEVENT

"""
duckdb.sql(sql)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [28]:
sql = """
SELECT
    pid,
    label,
    project,
    otype
FROM out
WHERE otype = 'SamplingEvent'
"""
display(duckdb.sql(sql).show(max_rows=100))

┌────────────────────────────────────────────────────┬───────────────────────────────────────┬────────────────────────────────────────────────────┬───────────────┐
│                        pid                         │                 label                 │                      project                       │     otype     │
│                      varchar                       │                varchar                │                      varchar                       │    varchar    │
├────────────────────────────────────────────────────┼───────────────────────────────────────┼────────────────────────────────────────────────────┼───────────────┤
│ sampevent_05846df8046077419d1a7f998da25eb482ef7d99 │ Sampling event for: SUERC-7919        │ Cross-referenced p3k14c                            │ SamplingEvent │
│ sampevent_7bd27ec1a00f8e9b35532df008e78ab711f24007 │ Sampling event for: BETA-230347       │ Cross-referenced p3k14c                            │ SamplingEvent │
│ sampevent_1ba3

None

In [29]:
# Relate the sampling sites with the sampling sites geolocations
make_s_p_o_edge_row(
    s_col='PID_SAMPSITE',
    p_val='site_location',
    p_col=None,
    o_col='PID_GEOLOC_SITE',
    source_tab='samps',
    group_by_cols='PID_SAMPSITE, PID_GEOLOC_SITE'
)

In [30]:
# Relate the Sampling events and material samples
make_s_p_o_edge_row(
    s_col='PID_SAMP', 
    p_val='produced_by',
    p_col=None,
    o_col='PID_SAMPEVENT',
    source_tab='samps',
    group_by_cols='PID_SAMP, PID_SAMPEVENT'
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [31]:
# Relate the Sampling events and sampling sites
make_s_p_o_edge_row(
    s_col='PID_SAMPEVENT', 
    p_val='sampling_site',
    p_col=None,
    o_col='PID_SAMPSITE',
    source_tab='samps',
    group_by_cols='PID_SAMPEVENT, PID_SAMPSITE'
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [32]:
# Relate the Sampling events and geolocations for the samples
make_s_p_o_edge_row(
    s_col='PID_SAMPEVENT', 
    p_val='sample_location',
    p_col=None,
    o_col='PID_GEOLOC_SAMP',
    source_tab='samps',
    group_by_cols='PID_SAMPEVENT, PID_GEOLOC_SAMP'
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [33]:
edge_checks = [
    'site_location',
    'sample_location',
    'sampling_site',
    'produced_by',
]
for e_check in edge_checks:
    sql = f"SELECT COUNT(pid), '{e_check}' AS predicate FROM out WHERE p = '{e_check}'"
    display(duckdb.sql(sql).show(max_rows=100))

┌────────────┬───────────────┐
│ count(pid) │   predicate   │
│   int64    │    varchar    │
├────────────┼───────────────┤
│      13777 │ site_location │
└────────────┴───────────────┘



None

┌────────────┬─────────────────┐
│ count(pid) │    predicate    │
│   int64    │     varchar     │
├────────────┼─────────────────┤
│    1050477 │ sample_location │
└────────────┴─────────────────┘



None

┌────────────┬───────────────┐
│ count(pid) │   predicate   │
│   int64    │    varchar    │
├────────────┼───────────────┤
│    1050555 │ sampling_site │
└────────────┴───────────────┘



None

┌────────────┬─────────────┐
│ count(pid) │  predicate  │
│   int64    │   varchar   │
├────────────┼─────────────┤
│    1050555 │ produced_by │
└────────────┴─────────────┘



None

In [34]:
sql = "SELECT count(pid) AS pcnt, pid FROM out WHERE otype = 'MaterialSampleRecord' GROUP BY pid ORDER BY pcnt DESC"
display(duckdb.sql(sql).show(max_rows=100))

┌───────┬─────────────────────────────────┐
│ pcnt  │               pid               │
│ int64 │             varchar             │
├───────┼─────────────────────────────────┤
│     1 │ ark:/28722/k2w66rh1b            │
│     1 │ ark:/28722/k2bg3046c            │
│     1 │ ark:/28722/k2t72tk18            │
│     1 │ ark:/28722/k2bg2rx2g            │
│     1 │ ark:/28722/r2p3k14c/hel_2564    │
│     1 │ ark:/28722/k2pc32n5h            │
│     1 │ ark:/28722/k2765gf9q            │
│     1 │ ark:/28722/r2p3k14c/wsu_2230    │
│     1 │ ark:/28722/k2gq77s02            │
│     1 │ ark:/28722/k23n2627m            │
│     1 │ ark:/28722/k2z32847k            │
│     1 │ ark:/28722/k27m0pg8d            │
│     1 │ ark:/28722/k27s8487c            │
│     1 │ ark:/28722/k24j0hm0n            │
│     1 │ ark:/28722/k2f47qv31            │
│     1 │ ark:/28722/k2gq6rc6d            │
│     1 │ ark:/28722/k2mp5938x            │
│     1 │ ark:/28722/r2p24/pc_19940152    │
│     1 │ ark:/28722/k2xd15d4w  

None

In [35]:
sql = "SELECT COUNT(pid), otype FROM out GROUP BY otype"
display(duckdb.sql(sql).show(max_rows=100))

┌────────────┬─────────────────────────┐
│ count(pid) │          otype          │
│   int64    │         varchar         │
├────────────┼─────────────────────────┤
│    3165364 │ _edge_                  │
│       1985 │ IdentifiedConcept       │
│     193996 │ GeospatialCoordLocation │
│    1050555 │ MaterialSampleRecord    │
│    1050555 │ SamplingEvent           │
│      13777 │ SamplingSite            │
└────────────┴─────────────────────────┘



None

In [36]:
node_types = {
  "Agent": {
    "name": "name VARCHAR DEFAULT NULL",
    "affiliation": "affiliation VARCHAR DEFAULT NULL",
    "contact_information": "contact_information VARCHAR DEFAULT NULL",
    "role": "role VARCHAR DEFAULT NULL",
    "label": "label VARCHAR DEFAULT NULL",
    "description": "description VARCHAR DEFAULT NULL"
  },
  "IdentifiedConcept": {
    "label": "label VARCHAR DEFAULT NULL",
    "scheme_name": "scheme_name VARCHAR DEFAULT NULL",
    "scheme_uri": "scheme_uri VARCHAR DEFAULT NULL",
    "description": "description VARCHAR DEFAULT NULL"
  },
  "GeospatialCoordLocation": {
    "elevation": "elevation VARCHAR DEFAULT NULL",
    "latitude": "latitude DOUBLE DEFAULT NULL",
    "longitude": "longitude DOUBLE DEFAULT NULL",
    "obfuscated": "obfuscated BOOLEAN ",
    "label": "label VARCHAR DEFAULT NULL",
    "description": "description VARCHAR DEFAULT NULL"
  },
  "SamplingSite": {
    "description": "description VARCHAR DEFAULT NULL",
    "label": "label VARCHAR DEFAULT NULL",
    "place_name": "place_name VARCHAR[]",
    "is_part_of": "is_part_of VARCHAR[]"
  },
  "SamplingEvent": {
    "label": "label VARCHAR DEFAULT NULL",
    "description": "description VARCHAR DEFAULT NULL",
    "has_feature_of_interest": "has_feature_of_interest VARCHAR DEFAULT NULL",
    "project": "project VARCHAR DEFAULT NULL",
    "result_time": "result_time VARCHAR DEFAULT NULL",
    "authorized_by": "authorized_by VARCHAR[]"
  },
  "MaterialSampleCuration": {
    "access_constraints": "access_constraints VARCHAR[]",
    "curation_location": "curation_location VARCHAR DEFAULT NULL",
    "description": "description VARCHAR DEFAULT NULL",
    "label": "label VARCHAR DEFAULT NULL"
  },
  "SampleRelation": {
    "description": "description VARCHAR DEFAULT NULL",
    "label": "label VARCHAR DEFAULT NULL",
    "relationship": "relationship VARCHAR DEFAULT NULL",
    "target": "target VARCHAR DEFAULT NULL"
  },
  "MaterialSampleRecord": {
    "label": "label VARCHAR DEFAULT NULL",
    "last_modified_time": "last_modified_time VARCHAR DEFAULT NULL",
    "description": "description VARCHAR DEFAULT NULL",
    "sample_identifier": "sample_identifier VARCHAR DEFAULT NULL",
    "alternate_identifiers": "alternate_identifiers VARCHAR[]",
    "sampling_purpose": "sampling_purpose VARCHAR DEFAULT NULL",
    "complies_with": "complies_with VARCHAR[]",
    "dc_rights": "dc_rights VARCHAR DEFAULT NULL"
  }
}

edge_fields = [
  "pid",
  "otype",
  "s",
  "p",
  "o",
  "n",
  "altids",
  "geometry"
]

literal_fields = [
  "authorized_by",
  "has_feature_of_interest",
  "affiliation",
  "sampling_purpose",
  "complies_with",
  "project",
  "alternate_identifiers",
  "relationship",
  "elevation",
  "sample_identifier",
  "dc_rights",
  "result_time",
  "contact_information",
  "latitude",
  "target",
  "role",
  "scheme_uri",
  "is_part_of",
  "scheme_name",
  "name",
  "longitude",
  "obfuscated",
  "curation_location",
  "last_modified_time",
  "access_constraints",
  "place_name",
  "description",
  "label",
  "pid",
  "otype",
  "s",
  "p",
  "o",
  "n",
  "altids",
  "geometry"
]


In [37]:


outpath = os.path.join(data_dir, 'oc_isamples_pqg.parquet')
sql = f"COPY (SELECT * FROM out ) TO '{outpath}' "
sql += "(FORMAT PARQUET, KV_METADATA {"
sql += "pqg_version: '0.2.0', "
sql += "pqg_primary_key: 'pid', "
sql += f"pqg_node_types: '{json.dumps(node_types)}', "
sql += f"pqg_edge_fields: '{json.dumps(edge_fields)}', "
sql += f"pqg_literal_fields: '{json.dumps(literal_fields)}' "
sql += '})'

duckdb.execute(sql)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [38]:
sql = """
SELECT
COUNT(pid) AS ss_count, s
FROM out
WHERE p = 'sampling_site'
GROUP BY s
ORDER BY ss_count DESC
"""

# Verify that each sampling event only has 1 sampling site
display(duckdb.sql(sql).show(max_rows=100))

┌──────────┬────────────────────────────────────────────────────┐
│ ss_count │                         s                          │
│  int64   │                      varchar                       │
├──────────┼────────────────────────────────────────────────────┤
│        1 │ sampevent_b1ab92b8197b6e41cc51d825ee526cb9d47c8d7b │
│        1 │ sampevent_ce87ad6211fc6d7907c1ef9298c9d21ce05da136 │
│        1 │ sampevent_c73bb20d08bb0af398b1dd8e7496d329856e4fc5 │
│        1 │ sampevent_b1b46bc7600dbbebb1740e97830edaf2a5b4a974 │
│        1 │ sampevent_9a8c26890d4d5782ea82b3f3427623ab7ca0c03b │
│        1 │ sampevent_8acfd458c8765b6c63290170f719f1b24fd085f3 │
│        1 │ sampevent_ad6ed5cae95343ed6211638b9ec46bcb38eed6b3 │
│        1 │ sampevent_33955506f61b185bf07f5db23a5d29cc10808696 │
│        1 │ sampevent_ac32667edad98cdd8420db84de14dce11bce2846 │
│        1 │ sampevent_22a27e81233132976ea01147842a7a84f227720b │
│        1 │ sampevent_91abf2d7e1d17c80b06431f0ea1373c359b05643 │
│        1

None

In [39]:
sql = """
SELECT
COUNT(pid) AS spby_count, s
FROM out
WHERE p = 'produced_by'
GROUP BY s
ORDER BY spby_count DESC
"""

# Verify that each sampling event only has 1 sampling site
display(duckdb.sql(sql).show(max_rows=100))


┌────────────┬────────────────────────────────────┐
│ spby_count │                 s                  │
│   int64    │              varchar               │
├────────────┼────────────────────────────────────┤
│          1 │ ark:/28722/k2hx1f34m               │
│          1 │ ark:/28722/k2jw8nk6z               │
│          1 │ ark:/28722/k27949023               │
│          1 │ ark:/28722/k24t6wb90               │
│          1 │ ark:/28722/k2zk58w1h               │
│          1 │ ark:/28722/k2fq9vd3g               │
│          1 │ ark:/28722/k23n22n44               │
│          1 │ ark:/28722/k28p63r7r               │
│          1 │ ark:/28722/k2697kh2z               │
│          1 │ ark:/28722/k2kh0x88p               │
│          1 │ ark:/28722/k2ff3s62f               │
│          1 │ ark:/28722/r2p3k14c/oxa_18599      │
│          1 │ ark:/28722/r2p3k14c/cu_413         │
│          1 │ ark:/28722/r2p3k14c/beta_53588     │
│          1 │ ark:/28722/k2q52n32s               │
│          1

None

In [40]:
sql = """
SELECT
COUNT(pid) AS ssl_count, s
FROM out
WHERE p = 'sample_location'
GROUP BY s
ORDER BY ssl_count DESC
"""

# Verify that each sampling event only has 1 sample location
display(duckdb.sql(sql).show(max_rows=100))

┌───────────┬────────────────────────────────────────────────────┐
│ ssl_count │                         s                          │
│   int64   │                      varchar                       │
├───────────┼────────────────────────────────────────────────────┤
│         1 │ sampevent_8ea898ec79374b00cf1c2723361b1da779e666ad │
│         1 │ sampevent_9d304c76d924ab566f6c3892f99dddc22f1d1638 │
│         1 │ sampevent_216c800e8831803252b70c56601728b05192a183 │
│         1 │ sampevent_0faa206c18deef741f6d41c01d85066431423385 │
│         1 │ sampevent_1e3eac59a2ce5e91a6d50d2b808b82ad3b22a1d7 │
│         1 │ sampevent_6b0b75ab8b78783af738c48046b4a2683f777e13 │
│         1 │ sampevent_6f39c8964b2218075c8e739c59744ddcfea1e1e2 │
│         1 │ sampevent_36e1e495d0ddb5b228978f5b89df9c62b27c1073 │
│         1 │ sampevent_6203fc34c0ef412fedddf7a180960d91d5c92e64 │
│         1 │ sampevent_94783f3a60c4a40a5eefb466b9457605de8905cc │
│         1 │ sampevent_f1f0c0fb14fc9a0859f980149bab6349f3470f

None